# Radial Cluster Figures

For each (CBSA, cluster) pair in `auto_cluster_tracts.csv`, produces one figure spanning all five
decennial years (1980–2020).  Each figure has five columns (years) and five rows:

| row | content |
|-----|---------|
| 0 | city map at true Albers coordinates, shaded by hop distance from the cluster center |
| 1 | radial collapse — r = edge-distance, angle = true angle from the medoid |
| 2 | radial collapse — r = edge-distance, angle evenly spread within each ring |
| 3 | tracts per ring (bar chart) |
| 4 | black share per ring, population-weighted (bar chart) |

Ring scale (`rmax`) and ring-count y-axis (`ymax`) are fixed across all five years for each
cluster figure so that contraction / expansion is visible rather than silently rescaled.

Imports plotting and graph logic from `radial-tracts`; only the data-loading and the city panel
are written locally (different paths, configurable cluster label).

In [137]:
import sys
from pathlib import Path
import importlib

EXPERIMENT_DIR = Path("/Users/maria/Documents/capy-bara/experiments/h4_t3_observed_diffusion")
sys.path.insert(0, str(EXPERIMENT_DIR))

import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.gridspec import GridSpec
import networkx as nx

import utils.style, utils.graph, utils.distance, utils.radial, utils.figures
for _m in [utils.style, utils.graph, utils.distance, utils.radial, utils.figures]:
    importlib.reload(_m)

from utils import style
from utils.graph import centroid
from utils.distance import bfs
from utils.radial import bearings, spread_angles, radial_coords, ring_profile
from utils.figures import panel_radial, panel_ring_counts, panel_ring_share, SHARE_VMIN, SHARE_VMAX

style.apply()

In [138]:
ROOT       = Path("/Users/maria/Documents/capy-bara")
GRAPHS_DIR = ROOT / "data" / "processed" / "dual_graphs"
GEO_DIR    = ROOT / "data" / "processed" / "clipped_geographies"
DATA_DIR   = ROOT / "experiments" / "h4_t3_observed_diffusion" / "data"
OUT_DIR    = ROOT / "experiments" / "h4_t3_observed_diffusion" / "figures"
OUT_DIR.mkdir(exist_ok=True)

CBSA_NAMES    = {1714000: "Chicago", 4260000: "Philadelphia"}
CLUSTER_NAMES = {"cluster_1": "Cluster 1", "cluster_2": "Cluster 2"}
YEARS         = [1980, 1990, 2000, 2010, 2020]

COLOR_MODE = "log_count"   # "share"  → Black share (0–1 fixed scale)
                       # "log_count" → log(Black population + 1), scale fixed across years per cluster

In [139]:
def load_graph(area_code, year):
    path = GRAPHS_DIR / str(year) / f"tracts_in_max_city_{area_code}_{year}_march_2020_vintage_orig.json"
    raw  = json.loads(path.read_text())
    assert not raw["directed"] and not raw["multigraph"]
    G = nx.Graph()
    G.graph.update(year=int(year), area_code=int(area_code))
    for attrs in raw["nodes"]:
        attrs = dict(attrs)
        G.add_node(attrs.pop("id"), **attrs)
    order = [n["id"] for n in raw["nodes"]]
    for node, neighbours in zip(order, raw["adjacency"], strict=True):
        for edge in neighbours:
            edge_attrs = {k: v for k, v in edge.items() if k != "id"}
            G.add_edge(node, edge["id"], **edge_attrs)
    return G


def load_cluster(area_code, year, cluster, G):
    """Cluster rows from auto_cluster_tracts.csv with graph node IDs attached."""
    df = pd.read_csv(
        DATA_DIR / "auto_cluster_tracts.csv",
        dtype={"gisjoin": str, "geoid": str},
    )
    df = df[(df.area_code == int(area_code)) & (df.year == int(year)) & (df.cluster == cluster)].reset_index(drop=True)
    assert len(df), f"no rows for area_code={area_code} year={year} cluster={cluster}"
    gisjoin_to_node = {G.nodes[n]["GISJOIN"]: n for n in G}
    df["node_id"] = df.gisjoin.map(gisjoin_to_node)
    missing = df[df.node_id.isna()]
    if not missing.empty:
        print(f"  WARNING: {len(missing)} tracts not found in {year} graph: {missing.gisjoin.tolist()[:5]}")
    return df.dropna(subset=["node_id"]).assign(node_id=lambda d: d.node_id.astype(int)).reset_index(drop=True)


def load_center(area_code, year, cluster, G):
    """Returns (node_id, center_gisjoin, cluster_title) from cluster_metrics.csv."""
    df = pd.read_csv(
        DATA_DIR / "cluster_metrics.csv",
        dtype={"center_gisjoin": str, "center_geoid": str},
    )
    row = df[(df.area_code == int(area_code)) & (df.year == int(year)) & (df.cluster == cluster)]
    assert len(row) == 1, f"expected one metrics row for {area_code}/{year}/{cluster}, got {len(row)}"
    row = row.iloc[0]
    gisjoin_to_node = {G.nodes[n]["GISJOIN"]: n for n in G}
    node = gisjoin_to_node.get(row.center_gisjoin)
    assert node is not None, f"center GISJOIN {row.center_gisjoin} absent from {area_code}/{year} graph"
    return node, row.center_gisjoin, row.cluster_title

In [140]:
# --- network map (kept for reference, currently replaced by choropleth in row 0) -------------

def panel_city(ax, G, d, root, cluster_nodes, cluster_label, pad_km=4.0):
    """City map at true Albers coordinates, shaded by BFS hop distance from root."""
    cl = np.array([centroid(G, n) for n in cluster_nodes]) / 1000
    (x0, y0), (x1, y1) = cl.min(axis=0) - pad_km, cl.max(axis=0) + pad_km

    segs = [[centroid(G, u) / 1000, centroid(G, v) / 1000] for u, v in G.edges()]
    ax.add_collection(LineCollection(segs, colors=style.GRIDLINE, lw=0.5, zorder=1))

    nodes  = [n for n in G if n in d]
    pts = np.array([centroid(G, n) for n in nodes]) / 1000
    hops = np.array([d[n] for n in nodes])
    inside = (pts[:, 0] >= x0) & (pts[:, 0] <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
    ax.scatter(
        pts[:, 0], pts[:, 1], c=hops, cmap=style.CMAP_HOPS,
        s=17, lw=0, zorder=3, vmin=0, vmax=int(hops[inside].max()),
    )
    ax.scatter(cl[:, 0], cl[:, 1], facecolors="none", edgecolors=style.SERIES[0],
               s=55, lw=0.7, zorder=4)
    o = centroid(G, root) / 1000
    ax.plot(*o, "*", ms=15, mfc="#eb6834", #style.SURFACE,
            mec="#eb6834",#style.INK,
            mew=1.2, zorder=6)

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")
    ax.axis("off")

    handles = [
        plt.Line2D([], [], marker="o", ls="", mfc="none", mec=style.SERIES[0],
                   ms=7, mew=0.8, label=cluster_label),
        plt.Line2D([], [], marker="*", ls="",
                   mfc="#eb6834",#style.SURFACE,
                   mec="#eb6834",#style.INK,
                   ms=10, mew=0.9, label="center tract"),
    ]
    ax.legend(handles=handles, loc="lower left", borderpad=0.2,
              handletextpad=0.5, fontsize=10)


# --- choropleth (active) ---------------------------------------------------------------------

_CHORO_CMAP = style.CMAP_SHARE

def panel_choropleth(ax, cbsa, year, cluster_df, center_gisjoin, value_col="black_share", vmin=0.0, vmax=1.0):
    """Choropleth of cluster tracts shaded by `value_col`, with cluster boundary and medoid star."""
    geo_path = GEO_DIR / str(year) / f"tracts_in_max_city_{cbsa}_{year}_march_2020_vintage.gpkg"
    gdf = gpd.read_file(geo_path)

    merge_cols = ["gisjoin", value_col] if value_col != "black_share" else ["gisjoin", "black_share", "log_count"]
    cluster_gdf = gdf.merge(
        cluster_df[["gisjoin", "black_share", "log_count"]],
        left_on="GISJOIN", right_on="gisjoin",
        how="inner",
    )
    choro_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cluster_gdf["face_color"] = cluster_gdf[value_col].apply(
        lambda x: _CHORO_CMAP(choro_norm(x))
    )
    cluster_gdf.plot(ax=ax, color=cluster_gdf["face_color"].tolist(),
                     edgecolor="grey", linewidth=0.2, zorder=2)

    # Cluster outer boundary
    gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
        ax=ax, color=style.INK, linewidth=1, zorder=3)

    # Medoid star
    medoid_geom = gdf.loc[gdf["GISJOIN"] == center_gisjoin, "geometry"]
    if not medoid_geom.empty:
        c = medoid_geom.iloc[0].centroid
        ax.scatter(c.x, c.y, marker="*", s=60,
                   color="#eb6834",
                   edgecolor="#eb6834",
                   linewidth=0.6, zorder=4)

    ax.set_aspect("equal")
    ax.axis("off")

In [141]:
def compute_year_data(cbsa, year, cluster):
    """Load and compute everything one year's column needs."""
    G  = load_graph(cbsa, year)
    df = load_cluster(cbsa, year, cluster, G)
    root, center_gisjoin, _ = load_center(cbsa, year, cluster, G)

    cluster_title = df["cluster_title"].iloc[0]

    df["log_count"] = np.log1p(df["black_population"])

    d = bfs(G, root)

    # Restrict to tracts reachable from the center; disconnected components
    # (component_count > 1 in cluster_metrics) silently drop out here.
    nodes = [n for n in df.node_id.tolist() if n in d]
    n_dropped = len(df) - len(nodes)
    if n_dropped:
        print(f"    {n_dropped} tracts unreachable from center (disconnected component)")

    share     = {n: s  for n, s  in zip(df.node_id, df.black_share)     if n in d}
    log_count = {n: lc for n, lc in zip(df.node_id, df.log_count)       if n in d}
    black     = {n: b  for n, b  in zip(df.node_id, df.black_population) if n in d}

    theta          = bearings(G, root, nodes)
    coords_bearing = radial_coords(d, theta, nodes)
    coords_spread  = radial_coords(d, spread_angles(d, theta, nodes), nodes)
    reach          = max(d[n] for n in nodes)
    profile        = ring_profile(d, nodes, weights=black, values=share)

    return dict(
        year=year, G=G, d=d, root=root,
        cluster_df=df, center_gisjoin=center_gisjoin, cluster_title=cluster_title,
        nodes=nodes, share=share, log_count=log_count,
        coords_bearing=coords_bearing,
        coords_spread=coords_spread,
        reach=reach, profile=profile)

In [142]:
def figure_cluster_grid(cbsa, cluster, year_data_list, out_path):
    """5-year × 6-row grid figure for one (cbsa, cluster).

    Row 3 is an empty gap row — its height_ratio controls the spacing between
    the spatial panels (rows 0-2) and the bar-chart profiles (rows 4-5).
    rmax and ymax are shared across all years so contraction / expansion is
    visible rather than silently rescaled.
    """
    style.apply()

    # --- color mode ----------------------------------------------------------
    if COLOR_MODE == "log_count":
        color_key  = "log_count"
        color_col  = "log_count"
        c_vmin     = 0.0
        c_vmax     = max(v for r in year_data_list for v in r["log_count"].values())
        cbar_label = "log(Black population + 1)"
    else:
        color_key  = "share"
        color_col  = "black_share"
        c_vmin, c_vmax = 0.0, 1.0
        cbar_label = "Black share of tract population"

    rmax = max(r["reach"] for r in year_data_list)
    raw_ymax = max(p["n_tracts"] for r in year_data_list for p in r["profile"])
    ymax = int(np.ceil(raw_ymax / 10) * 10)

    city_name     = CBSA_NAMES[cbsa]
    cluster_label = year_data_list[0]["cluster_title"]
    n_years       = len(year_data_list)

    fig = plt.figure(figsize=(19.5, 15.0))
    gs  = GridSpec(
        6, n_years, figure=fig,
        height_ratios=[1.55, 1.2, 1.2, 0.30, 0.8, 0.8],  # row 3 = gap
        hspace=0.40, wspace=0.09,
        left=0.055, right=0.975, top=0.895, bottom=0.055,
    )

    scatter_ref = None
    row_ax = {}  # first-column axis per content row, for labels and colorbar

    for col, r in enumerate(year_data_list):
        year  = r["year"]
        first = col == 0

        # --- row 0: choropleth -----------------------------------------------
        ax = fig.add_subplot(gs[0, col])
        panel_choropleth(ax, cbsa, year, r["cluster_df"], r["center_gisjoin"],
                         value_col=color_col, vmin=c_vmin, vmax=c_vmax)
        ax.set_title(
            f"{year}\n{len(r['nodes'])} tracts, max edge distance: {r['reach']}",
            fontsize=10)
        if first: row_ax[0] = ax

        # --- row 1: radial bearing -------------------------------------------
        ax = fig.add_subplot(gs[1, col])
        sc = panel_radial(
            ax, r["coords_bearing"], r[color_key], rmax, r["reach"],
            title="Radial position = at edge-distance, real angle from medoid" if first else "",
            vmin=c_vmin, vmax=c_vmax)
        if scatter_ref is None:
            scatter_ref = sc
        if first: row_ax[1] = ax

        # --- row 2: radial spread --------------------------------------------
        ax = fig.add_subplot(gs[2, col])
        panel_radial(
            ax, r["coords_spread"], r[color_key], rmax, r["reach"],
            title="Radial position = at edge-distance, even spread" if first else "",
            vmin=c_vmin, vmax=c_vmax)
        if first: row_ax[2] = ax

        # row 3 is the gap — no subplot

        # --- row 4: ring counts ----------------------------------------------
        ax = fig.add_subplot(gs[4, col])
        panel_ring_counts(ax, r["profile"], rmax, ymax)
        if not first:
            ax.set_ylabel("")
            ax.set_yticklabels([])
        if first: row_ax[4] = ax

        # --- row 5: ring share -----------------------------------------------
        ax = fig.add_subplot(gs[5, col])
        panel_ring_share(ax, r["profile"], rmax)
        if not first:
            ax.set_ylabel("")
            ax.set_yticklabels([])
        if first: row_ax[5] = ax

    # Row labels in the left margin, centred on each row's axes
    row_label_map = [
        ("choropleth\n(Black share)" if COLOR_MODE == "share" else "choropleth\n(log count)", 0),
        ("radial\n(same angle)",      1),
        ("radial\n(even spread)",     2),
        ("tracts\nper ring",          4),
        ("Black share\nper ring",     5),
    ]
    for label, row_idx in row_label_map:
        pos = row_ax[row_idx].get_position()
        fig.text(0.018, (pos.y0 + pos.y1) / 2, label,
                 ha="center", va="center", fontsize=10,
                 color=style.INK_SECONDARY, rotation=90, linespacing=1.4)

    # Colorbar centred in the gap row (between rows 2 and 4)
    pos2    = row_ax[2].get_position()
    pos4    = row_ax[4].get_position()
    gap_mid = (pos2.y0 + pos4.y1) / 2
    cbar_h  = 0.011
    cax = fig.add_axes([0.20, gap_mid - cbar_h / 2, 0.60, cbar_h])
    cb  = fig.colorbar(scatter_ref, cax=cax, orientation="horizontal")
    cb.set_label(cbar_label, color=style.INK_SECONDARY, fontsize=10)
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=0, labelsize=8, labelcolor=style.INK_SECONDARY)

    fig.suptitle(
        f"{cluster_label},  1980-2020",
        x=0.055, y=0.965, ha="left", fontsize=14, color=style.INK)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, facecolor=style.SURFACE)
    plt.close(fig)
    print(f"  -> {out_path}")
    return out_path

In [143]:
for area_code in CBSA_NAMES:
    for cluster in CLUSTER_NAMES:
        year_data_list = []
        for year in YEARS:
            print(f"  {year}...", end=" ", flush=True)
            r = compute_year_data(area_code, year, cluster)
            print(f"{len(r['nodes'])} tracts  reach={r['reach']}")
            year_data_list.append(r)

        out_path = OUT_DIR / f"radial_{year_data_list[0]["cluster_title"].lower().replace(' ', '_').replace(',', '')[:30]}.png"
        figure_cluster_grid(area_code, cluster, year_data_list, out_path)

  1980... 260 tracts  reach=13
  1990... 260 tracts  reach=14
  2000... 260 tracts  reach=14
  2010... 236 tracts  reach=10
  2020... 230 tracts  reach=10


/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.


  -> /Users/maria/Documents/capy-bara/experiments/h4_t3_observed_diffusion/figures/radial_chicago_south_side.png
  1980... 121 tracts  reach=12
  1990... 121 tracts  reach=12
  2000... 121 tracts  reach=13
  2010... 85 tracts  reach=7
  2020... 84 tracts  reach=7


/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.


  -> /Users/maria/Documents/capy-bara/experiments/h4_t3_observed_diffusion/figures/radial_chicago_austin.png
  1980... 118 tracts  reach=9
  1990... 119 tracts  reach=9
  2000... 121 tracts  reach=9
  2010... 129 tracts  reach=9
  2020... 132 tracts  reach=9


/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.


  -> /Users/maria/Documents/capy-bara/experiments/h4_t3_observed_diffusion/figures/radial_philadelphia_germantown.png
  1980... 63 tracts  reach=15
  1990... 63 tracts  reach=14
  2000... 64 tracts  reach=7
  2010... 62 tracts  reach=6
  2020... 63 tracts  reach=7


/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gpd.GeoSeries([cluster_gdf.geometry.unary_union], crs=cluster_gdf.crs).boundary.plot(
/var/folders/l9/fbsr1cgs4qn42tbdhy2dsbz40000gn/T/ipykernel_90994/2012089849.py:66: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.


  -> /Users/maria/Documents/capy-bara/experiments/h4_t3_observed_diffusion/figures/radial_philadelphia_chester.png
